In [1]:
import getpass
import os
from dotenv import load_dotenv
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector = embeddings.embed_query("hello, world!")
vector[:5]

[-0.02276923693716526,
 0.010134130716323853,
 0.011886735446751118,
 -0.09669032692909241,
 -0.0027089761570096016]

In [5]:
from pypdf import PdfReader

In [6]:
reader = PdfReader("combined.pdf")
content = "".join([p.extract_text() for p in reader.pages])

In [21]:
from uuid import uuid4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
document = Document(
    page_content=content,
    metadata={"source": "dataquality_framework"},
)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100000, chunk_overlap=0)

chunk_list = text_splitter.split_documents([document])

chunk_document =[]
id_list =[]
for i in range(len(chunk_list)):
    uid = uuid4()
    chunk_document.append(Document(page_content=chunk_list[i].page_content,metadata={"source": "dataquality_framework"},id=str(uid)))
    id_list.append(str(uid))

In [17]:
chunk_document

[Document(id='99fb7b54-7a1a-4d1b-aa6d-755a17965b26', metadata={'source': 'dataquality_framework'}, page_content='Unclassified - Non -Classifié  \nData Quality Blog Post  \nIntroduction  \nWhy is it important to have  data quality  tests? To start with an analogy , cars (the end product)  must \ngo through rigorous tests and inspections  (quality checks)  from the manufacturer  before they are \nready to be sold on the market . We are applying similar standards to the datasets  – our end product  \n– on the data portal, as the data shared by Fisheries and Oceans Canada (DFO) comes from our \ninternal data collect ion processes , or the manufacturing of dat a following the logic of the analogy,  \nbefore they can be shared with the public.  \nBefore finalization of a dataset , such as after data collection  is done by a  scientists that work at \nDFO for example , their data will be tested following the methodologies described below  as a \nprocess of quality control . These tests will b

In [18]:
# alluid = [uuid4() for _ in range(len(
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)


In [22]:
id_list

['d1a0f8f1-c0c8-4cac-b41a-1faf494ca659']

In [23]:
vector_store.add_documents(documents=chunk_document, ids=id_list)

['d1a0f8f1-c0c8-4cac-b41a-1faf494ca659']

In [26]:
print(chunk_list[0].page_content)

Unclassified - Non -Classifié  
Data Quality Blog Post  
Introduction  
Why is it important to have  data quality  tests? To start with an analogy , cars (the end product)  must 
go through rigorous tests and inspections  (quality checks)  from the manufacturer  before they are 
ready to be sold on the market . We are applying similar standards to the datasets  – our end product  
– on the data portal, as the data shared by Fisheries and Oceans Canada (DFO) comes from our 
internal data collect ion processes , or the manufacturing of dat a following the logic of the analogy,  
before they can be shared with the public.  
Before finalization of a dataset , such as after data collection  is done by a  scientists that work at 
DFO for example , their data will be tested following the methodologies described below  as a 
process of quality control . These tests will be applied to  all datasets that are shared on the Salmon 
Data Portal, ensuring that the datasets are high-quality, reliable

In [27]:
vector_store.save_local("vectorstore/faiss_index")

In [28]:
new_vector_store = FAISS.load_local(
    "vectorstore/faiss_index", embeddings, allow_dangerous_deserialization=True
)

In [29]:
docs = new_vector_store.similarity_search("uniqueness")

In [30]:
docs

[Document(id='d1a0f8f1-c0c8-4cac-b41a-1faf494ca659', metadata={'source': 'dataquality_framework'}, page_content='Unclassified - Non -Classifié  \nData Quality Blog Post  \nIntroduction  \nWhy is it important to have  data quality  tests? To start with an analogy , cars (the end product)  must \ngo through rigorous tests and inspections  (quality checks)  from the manufacturer  before they are \nready to be sold on the market . We are applying similar standards to the datasets  – our end product  \n– on the data portal, as the data shared by Fisheries and Oceans Canada (DFO) comes from our \ninternal data collect ion processes , or the manufacturing of dat a following the logic of the analogy,  \nbefore they can be shared with the public.  \nBefore finalization of a dataset , such as after data collection  is done by a  scientists that work at \nDFO for example , their data will be tested following the methodologies described below  as a \nprocess of quality control . These tests will b

In [31]:
docs[0].page_content

'Unclassified - Non -Classifié  \nData Quality Blog Post  \nIntroduction  \nWhy is it important to have  data quality  tests? To start with an analogy , cars (the end product)  must \ngo through rigorous tests and inspections  (quality checks)  from the manufacturer  before they are \nready to be sold on the market . We are applying similar standards to the datasets  – our end product  \n– on the data portal, as the data shared by Fisheries and Oceans Canada (DFO) comes from our \ninternal data collect ion processes , or the manufacturing of dat a following the logic of the analogy,  \nbefore they can be shared with the public.  \nBefore finalization of a dataset , such as after data collection  is done by a  scientists that work at \nDFO for example , their data will be tested following the methodologies described below  as a \nprocess of quality control . These tests will be applied to  all datasets that are shared on the Salmon \nData Portal, ensuring that the datasets are high-qual